<a href="https://colab.research.google.com/github/ant-jan/text-networks-workshop-26/blob/main/notebooks/text_networks_workbook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Text Derived Network Analysis Workshop

This workbook introduces network analysis from text using *Alice’s Adventures in Wonderland* as a hands-on example. The structure of this workshop is inspired by Thomas T. Hills’ *Behavioral Network Science: Language, Mind, and Society* (2025).

We will turn a literary text into a character network, then use the same logic to think about psychological text data, such as emotions, symptoms, themes, or concepts in open-ended responses.

## Workshop workflow

1. Setup (quick)
2. Load the text (quick)
3. Clean and inspect the text (quick)
4. Split the book into chapters (quick)
5. Identify character appearances
6. Create an affiliation matrix
7. Build a bipartite network
8. Project it into a one-mode character network
9. Visualize and threshold the network
10. Calculate basic network measures
11. Interpret the network and translate the method to psychology


# 1. Setup

First, we prepare the R environment.

We clear the workspace, install any missing packages, and load the packages we will use throughout the workshop.

In [5]:
# ---- Setup ----

rm(list = ls())

packages <- c(
  "igraph",
  "dplyr",
  "tidyr",
  "tibble",
  "stringr"
)

installed <- rownames(installed.packages())

for (pkg in packages) {
  if (!pkg %in% installed) {
    install.packages(pkg, repos = "https://cloud.r-project.org")
  }
}

library(igraph)
library(dplyr)
library(tidyr)
library(tibble)
library(stringr)

# 2. Load the text

We will read the text file directly from GitHub.

The file is a plain-text version of *Alice’s Adventures in Wonderland* from Project Gutenberg. At this stage, it still includes Project Gutenberg header and footer material, so we will inspect it before cleaning.

In [6]:
# ---- Load Alice text from GitHub ----

alice_url <- "https://raw.githubusercontent.com/ant-jan/text-networks-workshop-26/main/data/raw/alice_in_wonderland.txt"

alice_raw <- readLines(alice_url, encoding = "UTF-8", warn = FALSE)

# Inspect the first 20 lines
writeLines(head(alice_raw, 40))

*** START OF THE PROJECT GUTENBERG EBOOK 11 ***

[Illustration]




Alice’s Adventures in Wonderland

by Lewis Carroll

THE MILLENNIUM FULCRUM EDITION 3.0

Contents

 CHAPTER I.     Down the Rabbit-Hole
 CHAPTER II.    The Pool of Tears
 CHAPTER III.   A Caucus-Race and a Long Tale
 CHAPTER IV.    The Rabbit Sends in a Little Bill
 CHAPTER V.     Advice from a Caterpillar
 CHAPTER VI.    Pig and Pepper
 CHAPTER VII.   A Mad Tea-Party
 CHAPTER VIII.  The Queen’s Croquet-Ground
 CHAPTER IX.    The Mock Turtle’s Story
 CHAPTER X.     The Lobster Quadrille
 CHAPTER XI.    Who Stole the Tarts?
 CHAPTER XII.   Alice’s Evidence




CHAPTER I.
Down the Rabbit-Hole


Alice was beginning to get very tired of sitting by her sister on the
bank, and of having nothing to do: once or twice she had peeped into
the book her sister was reading, but it had no pictures or
conversations in it, “and what is the use of a book,” thought Alice
“without pictures or conversations?”


# 3. Clean the text

The raw file includes Project Gutenberg metadata before and after the actual book.

Before analysing the text, we remove the header and footer so that our network is based only on the story.

In [16]:
# ---- Clean the text ----

# Start at the actual beginning of the story
start_line <- str_which(alice_raw, "^CHAPTER I\\.$")[1]

# End at "THE END"
end_line <- str_which(alice_raw, "^THE END$")[1]

# Keep only the story text, excluding "THE END"
alice_text <- alice_raw[start_line:(end_line - 1)]

# Remove illustration markers and empty lines
alice_text <- alice_text[!str_detect(alice_text, "^\\[Illustration\\]$")]
alice_text <- alice_text[str_trim(alice_text) != ""]

# Inspect the cleaned text
writeLines(head(alice_text, 2))
writeLines(tail(alice_text, 2))

CHAPTER I.
Down the Rabbit-Hole
their simple joys, remembering her own child-life, and the happy summer
days.


# 4. Split the book into chapters

Our first unit of analysis will be the chapter.

Later, when two characters are connected in the network, it will mean that they appeared in the same chapter. This is a methodological choice: we could also use scenes, paragraphs, or fixed-size text windows.


In [17]:
# ---- Split the book into chapters ----

# Find chapter heading lines
chapter_lines <- str_which(alice_text, "^CHAPTER [IVXLC]+\\.$")

# Inspect chapter headings
alice_text[chapter_lines]

[1] "CHAPTER I."    "CHAPTER II."   "CHAPTER III."  "CHAPTER IV."  
 [5] "CHAPTER V."    "CHAPTER VI."   "CHAPTER VII."  "CHAPTER VIII."
 [9] "CHAPTER IX."   "CHAPTER X."    "CHAPTER XI."   "CHAPTER XII."

In [22]:
# ---- Create chapter-level dataset ----

chapter_starts <- chapter_lines
chapter_ends <- c(chapter_lines[-1] - 1, length(alice_text))

chapters <- tibble(
  chapter_id = paste0("chapter_", seq_along(chapter_starts)),
  chapter_title = alice_text[chapter_starts + 1],
  text = mapply(
    function(start, end) paste(alice_text[(start + 2):end], collapse = " "),
    chapter_starts,
    chapter_ends
  )
)

#print one row of the created tibble
chapters %>%
  slice(1)

chapter_id chapter_title       
1 chapter_1  Down the Rabbit-Hole
  text                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                        

# 5. Identify character appearances

We now create a character dictionary: a list of characters and the words or phrases we will search for in each chapter.

This step matters because the network depends on our coding choices. For example, should "Rabbit" count as "White Rabbit"? Should "Cat" count as "Cheshire Cat"? These are interpretive decisions.

In [ ]:
# ---- Character dictionary ----

characters <- tibble(
  character = c(
    "Alice",
    "White Rabbit",
    "Mouse",
    "Dodo",
    "Lory",
    "Eaglet",
    "Caterpillar",
    "Duchess",
    "Cheshire Cat",
    "Hatter",
    "March Hare",
    "Dormouse",
    "Queen of Hearts",
    "King of Hearts",
    "Gryphon",
    "Mock Turtle",
    "Knave of Hearts",
    "Cook",
    "Bill"
  ),
  pattern = c(
    "\\bAlice\\b",
    "White Rabbit|\\bRabbit\\b",
    "\\bMouse\\b",
    "\\bDodo\\b",
    "\\bLory\\b",
    "\\bEaglet\\b",
    "\\bCaterpillar\\b",
    "\\bDuchess\\b",
    "Cheshire Cat|\\bCat\\b",
    "\\bHatter\\b",
    "March Hare|\\bHare\\b",
    "\\bDormouse\\b",
    "Queen of Hearts|\\bQueen\\b",
    "King of Hearts|\\bKing\\b",
    "\\bGryphon\\b",
    "Mock Turtle",
    "Knave of Hearts|\\bKnave\\b",
    "\\bCook\\b",
    "\\bBill\\b"
  )
)

characters